## load_fred
Loads the 10 FRED series CSVs from `RAW_FRED` (`/Volumes/{CATALOG}/raw/fred/`) into the single all-STRING Bronze table `{BRONZE}.fred_series_observations`. Each file is one series of `date,value,realtime_start,realtime_end`; the **`series_id` is not in the file** — it is injected per-file from the download manifest (`data_fetch.manifest.SOURCES`, keyed by landed filename). The source `date` column is renamed to `observation_date` (positional schema).

**Write strategy (A):** MERGE on `(series_id, observation_date, realtime_start)` — `realtime_start` is in the key to preserve FRED vintages. Each file holds one series, so per-file MERGE keeps source keys unique. **No archiving** (§18 deviation from §10 cell 7). DDL: `libs/ddl/bronze_ddl.py`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_FRED, PIPELINE_RUN_ID, STATUS_*, StepLog, Utils,
# ingestion_log_insert, spark, dbutils, F, StructType/StructField/StringType. data_fetch is
# importable via shared_lib_path (same mechanism download_sources uses).
from data_fetch.manifest import SOURCES

STEP_SEQUENCE = 1                                   # position is owned by the orchestrator
SOURCE_SYSTEM = "fred"
SOURCE_PATH   = RAW_FRED                            # /Volumes/{CATALOG}/raw/fred/
TARGET_TABLE  = f"{BRONZE}.fred_series_observations"

# landed_filename -> series_id, from the manifest (single source of truth for the mapping).
_fred_spec = next(s for s in SOURCES if s.name == SOURCE_SYSTEM)
FILE_TO_SERIES = {f.landed_filename: f.series_id for f in _fred_spec.files}

# Exact source header (4 cols) for validation; read schema renames date -> observation_date
# positionally (header=true skips row 1, schema applied by position).
EXPECTED_SOURCE_COLS = ["date", "value", "realtime_start", "realtime_end"]
read_schema = StructType([
    StructField("observation_date", StringType(), True),
    StructField("value",            StringType(), True),
    StructField("realtime_start",   StringType(), True),
    StructField("realtime_end",     StringType(), True),
])

MERGE_KEYS = ["series_id", "observation_date", "realtime_start"]

In [ ]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)
print(f"load_fred: step_log_id={step.step_log_id}")

In [ ]:
# Per-file header validation; also assert every file maps to a known series_id (fail loud on
# an unexpected filename). No-files CHECK inside the try; EXIT outside it (§10.1).
no_files = False
try:
    files = sorted(f.path for f in dbutils.fs.ls(SOURCE_PATH) if f.path.lower().endswith(".csv"))
    no_files = not files
    if not no_files:
        bad_files, unknown = [], []
        for file_path in files:
            basename = file_path.rstrip("/").split("/")[-1]
            if basename not in FILE_TO_SERIES:
                unknown.append(basename)
            actual = (
                spark.read.format("csv").option("header", "true")
                .load(file_path).limit(0).columns
            )
            if actual != EXPECTED_SOURCE_COLS:
                bad_files.append((file_path, actual))
        if unknown:
            raise ValueError(f"[{TARGET_TABLE}] {len(unknown)} file(s) not in the FRED "
                             f"manifest: {unknown}")
        if bad_files:
            raise ValueError(
                f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).\n"
                f"Expected: {EXPECTED_SOURCE_COLS}\n"
                + "\n".join(f"  {p}\n    actual: {h}" for p, h in bad_files)
            )
        print(f"load_fred: {len(files)} file(s) passed header + manifest validation.")
except Exception as e:
    step.fail(e); raise

if no_files:
    step.no_files()
    dbutils.notebook.exit(f"No CSV files found at {SOURCE_PATH}")

In [ ]:
# Read + shape + MERGE each series file. series_id injected as a literal from the manifest.
try:
    on_clause = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEYS)
    total_read = total_inserted = total_updated = 0

    for file_path in files:
        basename  = file_path.rstrip("/").split("/")[-1]
        series_id = FILE_TO_SERIES[basename]
        shaped_df = (
            spark.read.format("csv").option("header", "true").option("delimiter", ",")
                .schema(read_schema).load(file_path)
                .withColumn("series_id", F.lit(series_id))
                .withColumn("source_file_path", F.col("_metadata.file_path"))
                .withColumn("inserted_ts", F.current_timestamp())
                .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
        )
        n_read = shaped_df.count()
        shaped_df.createOrReplaceTempView("fred_staging")

        pre_count = spark.table(TARGET_TABLE).count()
        metrics = spark.sql(f"""
            MERGE INTO {TARGET_TABLE} AS t
            USING fred_staging AS s
            ON {on_clause}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """).first().asDict()
        post_count = spark.table(TARGET_TABLE).count()

        inserted = metrics.get("num_inserted_rows")
        if inserted is None:
            inserted = post_count - pre_count
        if post_count - pre_count != inserted:
            raise AssertionError(
                f"[{TARGET_TABLE}] Insert-count mismatch for {series_id}: MERGE reported "
                f"{inserted:,} inserts, row count grew by {post_count - pre_count:,}."
            )
        total_read     += n_read
        total_inserted += inserted
        total_updated  += metrics.get("num_updated_rows") or 0
        print(f"load_fred: {series_id}: read={n_read:,} inserted={inserted:,} "
              f"updated={metrics.get('num_updated_rows')}")

    step.rows_read    = total_read
    step.rows_written = total_inserted
    step.succeed()
    print(f"load_fred: DONE {len(files)} series read={total_read:,} "
          f"inserted={total_inserted:,} updated={total_updated:,}")
except Exception as e:
    step.fail(e); raise

# ingestion_log after succeed(), outside the try — one row per ingested file.
files_df = spark.createDataFrame([(p,) for p in files], "source_file_path string")
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_fred: WARNING ingestion_log insert failed: {res['error_message']}")